RAG 예제: "백과사전 질문-답변기

In [1]:
!pip install openai sentence-transformers chromadb python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 118.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [3]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import os
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-4o-mini"

In [4]:
# 1. 지식 구성 ----------------------------
# RAG에서 검색 대상이 되는 간단한 지식 문서들
knowledge = [
    "기린은 목이 길다.",
    "코끼리는 귀가 크고 무겁다.",
    "치타는 육상에서 가장 빠른 동물이다.",
    "하마는 물속에서 생활하는 포유류다.",
    "펭귄은 날지 못하지만 수영을 잘한다."
]
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # 2. 임베딩 모델 로드

# 지식 문서들을 임베딩 벡터로 변환
# normalize_embeddings=True를 사용하면 벡터 크기가 1로 정규화됨
# 정규화된 벡터끼리 내적하면 cosine similarity로 해석하기 쉬움
embeddings = embedder.encode(
    knowledge,
    normalize_embeddings=True
)
print("문서 임베딩:", embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

문서 임베딩: [[-0.04507371  0.12638244  0.07124437 ...  0.0676651  -0.05525566
  -0.02433011]
 [-0.02226256  0.08212922  0.08384718 ...  0.09879503  0.02848517
   0.01256627]
 [-0.0334919   0.09172806  0.00165735 ...  0.01161214 -0.06550936
  -0.00087902]
 [-0.01816243  0.06918667  0.05138754 ...  0.08816178 -0.07515938
  -0.00698429]
 [-0.04773956  0.16576874  0.03402462 ...  0.10616168 -0.03843705
   0.00790475]]


In [5]:
# 3. Chroma 저장소 구성 ----------------------------------
# ChromaDB를 로컬 디스크에 저장하는 벡터 데이터베이스 클라이언트 생성
chroma_client = chromadb.Client(Settings(
    persist_directory="./rag_demo",
    anonymized_telemetry=False   # 익명 사용 정보 전송 비활성화(권장)
))

# 실습에서는 같은 ID가 중복 추가될 수 있으므로 기존 컬렉션을 삭제하고 다시 생성
try:
    chroma_client.delete_collection("animals")
except:
    pass

# cosine 거리 기준으로 animals 컬렉션 생성
collection = chroma_client.create_collection(
    name="animals",
    metadata={"hnsw:space": "cosine"}
)

# 문서와 임베딩 벡터를 ChromaDB에 저장
for i, (text, emb) in enumerate(zip(knowledge, embeddings)):
    collection.add(
        documents=[text],
        embeddings=[emb.tolist()],
        ids=[f"doc_{i}"]
    )

# 저장된 문서와 ID 확인
all_data = collection.get()
print("\n저장된 전체 데이터:", all_data)
print("\n컬렉션에 저장된 문서 개수:", collection.count())

# 특정 ID의 문서만 조회
doc = collection.get(ids=["doc_0"])
print("\n특정 문서 조회:", doc)


저장된 전체 데이터: {'ids': ['doc_0', 'doc_1', 'doc_2', 'doc_3', 'doc_4'], 'embeddings': None, 'documents': ['기린은 목이 길다.', '코끼리는 귀가 크고 무겁다.', '치타는 육상에서 가장 빠른 동물이다.', '하마는 물속에서 생활하는 포유류다.', '펭귄은 날지 못하지만 수영을 잘한다.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None, None, None, None, None]}

컬렉션에 저장된 문서 개수: 5

특정 문서 조회: {'ids': ['doc_0'], 'embeddings': None, 'documents': ['기린은 목이 길다.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None]}


In [6]:
# 4. 질문 처리 ---------
query = "목이 긴 동물은?"    # 사용자의 질문

# 질문을 임베딩 벡터로 변환
query_vec = embedder.encode(
    [query],
    normalize_embeddings=True
)[0]

# 질문 벡터와 가까운 문서를 ChromaDB에서 검색
results = collection.query(
    query_embeddings=[query_vec.tolist()],
    n_results=3,
    include=["documents", "distances"]
)
print("\n검색 결과:", results)

# 검색된 문서들을 하나의 참고 문맥으로 합치기
context = "\n".join(results["documents"][0])
print("\n검색된 참고 문서:", context)


검색 결과: {'ids': [['doc_0', 'doc_2', 'doc_4']], 'embeddings': None, 'documents': [['기린은 목이 길다.', '치타는 육상에서 가장 빠른 동물이다.', '펭귄은 날지 못하지만 수영을 잘한다.']], 'uris': None, 'included': ['documents', 'distances'], 'data': None, 'metadatas': None, 'distances': [[0.18601900339126587, 0.27172964811325073, 0.34308648109436035]]}

검색된 참고 문서: 기린은 목이 길다.
치타는 육상에서 가장 빠른 동물이다.
펭귄은 날지 못하지만 수영을 잘한다.


In [8]:

# 5. GPT에 연결하여 답변 생성 --------------------------
# 검색된 문서와 사용자 질문을 함께 프롬프트에 넣음
# 이것이 RAG에서 Generation 단계로 넘어가는 핵심 부분
prompt = f"""
아래 정보를 참고해서 질문에 답해줘.
질문 정보를 바탕으로 질문에 대해 상세하게 설명해 줘.
가능하면 예시나 관련 배경지식도 함께 덧붙여.

[참고 정보]
{context}

[질문]
{query}

추가 사항:
마크다운 기호 없이 평문으로 답해줘.
"""

print("프롬프트 확인 ---")
print(prompt)

프롬프트 확인 ---

아래 정보를 참고해서 질문에 답해줘.
질문 정보를 바탕으로 질문에 대해 상세하게 설명해 줘.
가능하면 예시나 관련 배경지식도 함께 덧붙여.

[참고 정보]
기린은 목이 길다.
치타는 육상에서 가장 빠른 동물이다.
펭귄은 날지 못하지만 수영을 잘한다.

[질문]
목이 긴 동물은?

추가 사항:
마크다운 기호 없이 평문으로 답해줘.



In [9]:

# GPT 모델에 프롬프트를 전달하여 답변 생성
response = client.responses.create(
    model=model,  input=prompt
)
print("\n답변:", response.output_text)


답변: 목이 긴 동물로는 기린이 가장 유명합니다. 기린은 길고 우아한 목을 가지고 있어, 이를 통해 나무의 높은 곳에 있는 잎을 쉽게 따 먹을 수 있습니다. 이러한 목의 길이는 기린이 두 개의 특정한 진화적 이점을 가지고 있기 때문입니다. 첫째, 먹이를 찾기 위한 경쟁에서 유리해지며, 둘째, 다른 기린들과의 싸움에서 상대방보다 높은 위치를 차지할 수 있습니다.

기린 외에도 여러 동물들이 목이 길다는 특징을 가지고 있지만, 기린처럼 극단적으로 긴 경우는 드물고, 일반적으로는 기린이 대표적으로 알려져 있습니다. 예를 들어, 일부 종류의 타라멘트(타라멘트와는 관련이 없지만 비슷한 신체적 특성을 가진 동물들도 목이 길어지는 경향이 있습니다)도 긴 목을 가지고 있습니다.

목이 길다는 것은 단순히 먹이를 얻기 위한 것이 아니라 여러 환경에서 생존 경쟁에서 유리한 점을 얻게 해주는 중요한 특징으로 작용할 수 있습니다.
